# Reranker yes/no decision (real pipeline example)

This notebook uses your **Qwen3 embedding + ChromaDB + Qwen3 reranker** pipeline
and then applies a decision layer on the reranker scores.

The reranker produces a score **per candidate**. You still need a rule to say:
- "match" (yes)
- "no match" (no)

We demonstrate two common rules:
1) **Absolute threshold**
2) **Margin (top1 - top2) threshold**

> Note: This runs real models. Adjust `SAMPLE_*` values for speed.


In [1]:
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm
import chromadb

from scripts.qwen3_vl_embedding import Qwen3VLEmbedder
from scripts.qwen3_vl_reranker import Qwen3VLReranker

# -----------------
# Config (from main.ipynb)
# -----------------
class Config:
    PRODUCTS_CSV = Path("dataset/products.csv")
    QUERIES_CSV = Path("dataset/queries.csv")
    IMAGES_DIR = Path("dataset/images")
    CHROMA_PATH = Path("chroma_db_final_v2")

    EMBEDDING_MODEL = "Qwen/Qwen3-VL-Embedding-2B"
    RERANKER_MODEL = "Qwen/Qwen3-VL-Reranker-2B"

    RETRIEVAL_TOP_K = 10
    BATCH_SIZE = 10

config = Config()

# -----------------
# Demo settings
# -----------------
SAMPLE_QUERIES = 5          # reduce for speed
INDEX_LIMIT = 200           # None = index all products (slow)

# Load data
products_df = pd.read_csv(config.PRODUCTS_CSV)
queries_df = pd.read_csv(config.QUERIES_CSV)

queries_sample = queries_df.sample(SAMPLE_QUERIES, random_state=42).reset_index(drop=True)

print(f"Products: {len(products_df)} | Queries sample: {len(queries_sample)}")

Products: 404 | Queries sample: 5


In [2]:
# -----------------
# Load embedding model
# -----------------
print(f"Loading embedding model: {config.EMBEDDING_MODEL}")
embedder = Qwen3VLEmbedder(
    model_name_or_path=config.EMBEDDING_MODEL,
    torch_dtype=torch.float16
)
print("Embedding model loaded")

# -----------------
# ChromaDB init (reuse if already populated)
# -----------------
client = chromadb.PersistentClient(path=str(config.CHROMA_PATH))
collection = client.get_or_create_collection(
    name="products",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB path: {config.CHROMA_PATH}")
print(f"Existing items in collection: {collection.count()}")

# -----------------
# Index products (only if empty)
# -----------------

def index_products(products_df, embedder, collection, images_dir, batch_size=10):
    print(f"Indexing {len(products_df)} products...")
    indexed = 0

    for i in tqdm(range(0, len(products_df), batch_size), desc="Indexing"):
        batch = products_df.iloc[i:i+batch_size]

        inputs = []
        ids = []
        docs = []
        metadatas = []

        for _, row in batch.iterrows():
            text = f"{row['name']} {str(row.get('description', ''))}"
            img_path = images_dir / row['image']

            if img_path.exists():
                inputs.append({"text": text, "image": str(img_path.absolute())})
            else:
                inputs.append({"text": text})

            ids.append(row['db_id'])
            docs.append(text)
            metadatas.append({
                "name": str(row['name']),
                "image": row['image'],
                "price": str(row.get('price', '')),
                "source_id": str(row.get('source_id', ''))
            })

        embeddings = embedder.process(inputs)
        embeddings_list = embeddings.float().cpu().numpy().tolist()

        collection.add(
            ids=ids,
            embeddings=embeddings_list,
            documents=docs,
            metadatas=metadatas
        )

        indexed += len(ids)

    print(f"Indexed {indexed} products")
    return indexed

if collection.count() == 0:
    products_to_index = products_df if INDEX_LIMIT is None else products_df.head(INDEX_LIMIT)
    index_products(products_to_index, embedder, collection, config.IMAGES_DIR, config.BATCH_SIZE)
else:
    print("Using existing ChromaDB collection")

Loading embedding model: Qwen/Qwen3-VL-Embedding-2B


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Embedding model loaded
ChromaDB path: chroma_db_final_v2
Existing items in collection: 404
Using existing ChromaDB collection


In [3]:
# -----------------
# Compute query embeddings (sample)
# -----------------

def compute_query_embeddings(queries_df, embedder, images_dir, batch_size=10):
    print(f"Computing embeddings for {len(queries_df)} queries...")
    all_embeddings = []

    for i in tqdm(range(0, len(queries_df), batch_size), desc="Query Embeddings"):
        batch = queries_df.iloc[i:i+batch_size]
        inputs = []
        for _, row in batch.iterrows():
            text = f"{row['name']} {str(row.get('description', ''))[:300]}"
            img_path = images_dir / row['image']

            if img_path.exists():
                inputs.append({"text": text, "image": str(img_path.absolute())})
            else:
                inputs.append({"text": text})

        embeddings = embedder.process(inputs)
        all_embeddings.extend(embeddings.float().cpu().numpy().tolist())

    print(f"Computed {len(all_embeddings)} embeddings")
    return all_embeddings

query_embeddings = compute_query_embeddings(
    queries_sample, embedder, config.IMAGES_DIR, config.BATCH_SIZE
)

# -----------------
# Retrieval (top-K candidates)
# -----------------

def retrieve_candidates(queries_df, query_embeddings, collection, top_k=10):
    print(f"Retrieving top-{top_k} candidates for {len(queries_df)} queries...")
    all_results = []

    for idx, (_, row) in enumerate(tqdm(queries_df.iterrows(), total=len(queries_df), desc="Retrieval")):
        results = collection.query(
            query_embeddings=[query_embeddings[idx]],
            n_results=top_k
        )

        candidates = []
        for i in range(len(results['ids'][0])):
            candidates.append({
                'db_id': results['ids'][0][i],
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i],
                'similarity': 1 - results['distances'][0][i]
            })

        all_results.append({
            'query_id': row['query_id'],
            'has_match': row['has_match'],
            'ground_truth_db_id': row['ground_truth_db_id'] if row['has_match'] else None,
            'candidates': candidates
        })

    print(f"Retrieved candidates for {len(all_results)} queries")
    return all_results

retrieval_results = retrieve_candidates(
    queries_sample, query_embeddings, collection, config.RETRIEVAL_TOP_K
)

Computing embeddings for 5 queries...


Query Embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]


Computed 5 embeddings
Retrieving top-10 candidates for 5 queries...


Retrieval: 100%|██████████| 5/5 [00:00<00:00, 88.35it/s]

Retrieved candidates for 5 queries


In [5]:
# -----------------
# Load reranker and rerank
# -----------------
print(f"Loading reranker model: {config.RERANKER_MODEL}")
reranker = Qwen3VLReranker(
    model_name_or_path=config.RERANKER_MODEL,
    torch_dtype=torch.float16
)
print("Reranker model loaded")

RERANKER_INSTRUCTION = (
    "Determine if these two products are EXACTLY the same item with same size, "
    "quantity, and specifications. Products with different pack sizes, quantities, "
    "or dimensions are NOT the same."
)


def rerank_all(queries_df, retrieval_results, reranker, images_dir):
    print(f"Reranking candidates for {len(retrieval_results)} queries...")
    all_reranked = []

    for idx, r in enumerate(tqdm(retrieval_results, desc="Reranking")):
        query_row = queries_df.iloc[idx]
        candidates = r['candidates'][:5]

        if not candidates:
            all_reranked.append({**r, 'reranked': [], 'best_score': 0.0})
            continue

        query_text = f"{query_row['name']} {str(query_row.get('description', ''))[:300]}"
        query_img_path = images_dir / query_row['image']

        query = {"text": query_text}
        if query_img_path.exists():
            query['image'] = str(query_img_path.absolute())

        documents = []
        for cand in candidates:
            doc = {"text": cand['text']}
            img_path = images_dir / cand['metadata'].get('image', '')
            if img_path.exists():
                doc['image'] = str(img_path.absolute())
            documents.append(doc)

        scores = reranker.process({
            "instruction": RERANKER_INSTRUCTION,
            "query": query,
            "documents": documents
        })

        reranked = []
        for cand, score in zip(candidates, scores):
            reranked.append({**cand, 'rerank_score': score})
        reranked.sort(key=lambda x: x['rerank_score'], reverse=True)

        all_reranked.append({
            **r,
            'reranked': reranked,
            'best_score': reranked[0]['rerank_score'] if reranked else 0.0
        })

    print(f"Reranked {len(all_reranked)} queries")
    return all_reranked

reranked_results = rerank_all(
    queries_sample, retrieval_results, reranker, config.IMAGES_DIR
)


Loading reranker model: Qwen/Qwen3-VL-Reranker-2B


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Reranker model loaded
Reranking candidates for 5 queries...


Reranking: 100%|██████████| 5/5 [01:04<00:00, 12.91s/it]

Reranked 5 queries


In [6]:
# -----------------
# Decision layer (yes/no) using real reranker scores
# -----------------
ABS_THRESHOLD = 0.75
MARGIN_THRESHOLD = 0.05

rows = []
for r in reranked_results:
    if not r['reranked']:
        rows.append({
            "query_id": r['query_id'],
            "top1_score": 0.0,
            "top2_score": 0.0,
            "abs_match": False,
            "margin_ok": False,
            "final_match": False,
            "has_match_label": r['has_match'],
        })
        continue

    top1 = r['reranked'][0]
    top2 = r['reranked'][1] if len(r['reranked']) > 1 else None

    top1_score = top1['rerank_score']
    top2_score = top2['rerank_score'] if top2 else 0.0

    abs_match = top1_score >= ABS_THRESHOLD
    margin_ok = (top1_score - top2_score) >= MARGIN_THRESHOLD
    final_match = abs_match and margin_ok

    rows.append({
        "query_id": r['query_id'],
        "top1_db_id": top1['db_id'],
        "top1_score": top1_score,
        "top2_score": top2_score,
        "abs_match": abs_match,
        "margin_ok": margin_ok,
        "final_match": final_match,
        "has_match_label": r['has_match'],
        "ground_truth_db_id": r.get('ground_truth_db_id')
    })

pd.DataFrame(rows)

,query_id,top1_db_id,top1_score,top2_score,abs_match,margin_ok,final_match,has_match_label,ground_truth_db_id
0,q_0070_match,db_0070,0.750488,0.442139,True,True,True,True,db_0070
1,q_0053_nomatch,db_0313,0.093018,0.074097,False,False,False,False,NaN
2,q_0218_match,db_0218,0.829102,0.117371,True,True,True,True,db_0218
3,q_0250_match,db_0250,0.702148,0.141602,False,True,False,True,db_0250
4,q_0039_match,db_0039,0.826660,0.139160,True,True,True,True,db_0039


## Interpretation

- The reranker **always returns a top-1 candidate** with a score.
- You decide "no match" by applying a **rule** (threshold, margin, or both).

Common options:
- **Strict**: `abs_match AND margin_ok` (higher precision)
- **Looser**: `abs_match OR margin_ok` (higher recall)

This is the missing "none-of-the-above" decision layer.
